In [1]:
# Install required engines silently
!pip install google-cloud-bigquery google-cloud-bigquery-storage pyarrow duckdb -q

In [ ]:
!pip install plotly -q

import plotly.express as px
import plotly.graph_objects as go

In [ ]:
import duckdb
import pyarrow as pa
from google.cloud import bigquery

# Initialize the BigQuery client pointing to your specific project
project_id = 'ecommerce-lakehouse-prod'
client = bigquery.Client(project=project_id)

# The Extraction Query (Leveraging your partition to save compute)
# We pull exactly 7 days of data for our EDA and feature engineering sandbox.
extract_query = """
    SELECT
        event_time,
        event_type,
        product_id,
        category_id,
        category_code,
        brand,
        price,
        user_id,
        user_session
    FROM `ecommerce-lakehouse-prod.ecommerce_analytics.stg_ecommerce_unified`
    WHERE event_time >= '2019-10-01' AND event_time < '2019-10-08'
"""

print("Initiating BigQuery extraction via Storage API...")

# Execute and pipe directly into PyArrow (bypassing pandas completely)
arrow_table = client.query(extract_query).to_arrow()

print(f"Extraction successful.")
print(f"In-memory Arrow table shape: {arrow_table.shape[0]:,} rows, {arrow_table.shape[1]} columns")

# Initialize the local DuckDB engine
con = duckdb.connect()

Initiating BigQuery extraction via Storage API...
Extraction successful.
In-memory Arrow table shape: 8,829,315 rows, 9 columns


cleaning

In [ ]:

f = """
    SELECT
            user_session,
            COUNT(*) AS total_events,
            MIN(event_time) AS session_start,
            MAX(event_time) AS session_end
        FROM arrow_table
        WHERE user_session IS NOT NULL
        GROUP BY user_session
"""


display(con.execute(f).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,user_session,total_events,session_start,session_end
0,00f65e1c-4bd3-45b2-bdc6-ed6ec55d1a6a,12,2019-10-06 11:53:40+00:00,2019-10-06 12:04:03+00:00
1,05e45e14-2567-462e-abe8-e1296f6d0839,3,2019-10-06 16:13:04+00:00,2019-10-06 16:14:12+00:00
2,00cb88bc-6abd-457c-8dcd-c83276120afc,3,2019-10-06 17:01:55+00:00,2019-10-06 17:03:16+00:00
3,06069119-6a16-4fcd-a636-80584ef1ab7b,2,2019-10-06 17:33:26+00:00,2019-10-06 17:36:00+00:00
4,011dac83-3e6b-4562-9fc4-12ea46ec3ab6,44,2019-10-06 05:05:19+00:00,2019-10-06 06:06:17+00:00
...,...,...,...,...
1877360,c204d939-7875-40c1-be40-e58ae40307d7,1,2019-10-02 22:07:52+00:00,2019-10-02 22:07:52+00:00
1877361,b2da54c6-9c04-431e-9598-a2ea7158212b,1,2019-10-02 04:53:34+00:00,2019-10-02 04:53:34+00:00
1877362,dc24438c-28b9-48f5-b687-8c140aba8393,1,2019-10-02 08:31:45+00:00,2019-10-02 08:31:45+00:00
1877363,c51b9a32-64ff-4476-a031-0fc5a94c7b94,1,2019-10-02 16:08:26+00:00,2019-10-02 16:08:26+00:00


In [ ]:
session_trace_query = """
    SELECT
        event_time,
        event_type,
        product_id,
        category_code,
        price,
        user_id
    FROM arrow_table
    WHERE user_session = '04abe478-1225-4394-92fc-edda87859303'
    ORDER BY event_time ASC
"""

print("Reconstructing chronological user journey...")
display(con.execute(session_trace_query).df())

Reconstructing chronological user journey...


,event_time,event_type,product_id,category_code,price,user_id
0,2019-10-06 16:41:33+00:00,view,1004767,electronics.smartphone,251.47,557432383
1,2019-10-06 16:44:59+00:00,view,1004767,electronics.smartphone,251.47,557432383
2,2019-10-06 16:45:23+00:00,view,1004870,electronics.smartphone,289.51,557432383
3,2019-10-06 16:46:19+00:00,view,1004870,electronics.smartphone,289.51,557432383
4,2019-10-06 16:53:40+00:00,view,1307073,computers.notebook,684.68,557432383
5,2019-10-06 16:56:22+00:00,view,1307336,computers.notebook,357.25,557432383
6,2019-10-06 16:58:19+00:00,view,1307336,computers.notebook,357.25,557432383
7,2019-10-06 16:58:32+00:00,view,1306591,computers.notebook,514.79,557432383


In [ ]:
# Query 2: Session Behavior and Time Deltas
session_time_query = """
    WITH session_base AS (
        SELECT
            user_session,
            COUNT(*) AS total_events,
            MIN(event_time) AS session_start,
            MAX(event_time) AS session_end
        FROM arrow_table
        WHERE user_session IS NOT NULL
        GROUP BY user_session
    )
    SELECT
        -- How many events happen in a typical session?
        ROUND(AVG(total_events), 1) AS avg_events_per_session,
        MAX(total_events) AS max_events_in_one_session,

        -- How long do these sessions last?
        ROUND(AVG(date_diff('second', session_start, session_end) / 60.0), 2) AS avg_duration_minutes,
        ROUND(MAX(date_diff('second', session_start, session_end) / 60.0), 2) AS max_duration_minutes,

        -- How many sessions were just 1 click (instant bounces)?
        COUNT(CASE WHEN total_events = 1 THEN 1 END) AS instant_bounce_sessions
    FROM session_base
"""

print("Executing Session Time Dynamics...")
display(con.execute(session_time_query).df())

Executing Session Time Dynamics...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,avg_events_per_session,max_events_in_one_session,avg_duration_minutes,max_duration_minutes,instant_bounce_sessions
0,4.7,1159,12.94,9788.5,640979


In [ ]:

# 1. Execute the query and capture the DataFrame
df_funnel = con.execute(funnel_price_query).df()

# --- CHART 1: The Interactive Conversion Funnel ---
print("Rendering Interactive Conversion Funnel...")

# We map the exact funnel structure. Plotly will automatically draw the drop-off.
fig_funnel = px.funnel(
    df_funnel,
    x='total_events',
    y='event_type',
    title='E-Commerce Conversion Funnel (Drop-off Analysis)',
    labels={'total_events': 'Total Volume', 'event_type': 'User Action'},
    color='event_type',
    template='plotly_dark'
)

# Make the numbers easy to read (e.g., 1.5M instead of 1500000)
fig_funnel.update_traces(textinfo="value+percent initial")
fig_funnel.show()


# --- CHART 2: Average Price Dynamics ---
print("Rendering Price Dynamics...")

# Are people buying the expensive items, or just viewing them?
fig_price = px.bar(
    df_funnel,
    x='event_type',
    y='average_price',
    text='average_price',
    title='Average Item Price at Each Funnel Stage',
    labels={'average_price': 'Average Price ($)', 'event_type': 'User Action'},
    color='event_type',
    template='plotly_dark'
)

# Format the text to look like actual currency
fig_price.update_traces(texttemplate='$%{text:,.2f}', textposition='outside')
fig_price.update_layout(yaxis=dict(title='Price in USD'))
fig_price.show()

Rendering Interactive Conversion Funnel...


Rendering Price Dynamics...


In [ ]:
null_check_query = """
    SELECT
        COUNT(*) AS total_events,

        -- Event Type Nulls
        SUM(CASE WHEN event_type IS NULL THEN 1 ELSE 0 END) AS null_event_type_count,
        ROUND((SUM(CASE WHEN event_type IS NULL THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 4) AS null_event_type_pct,

        -- User Session Nulls
        SUM(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END) AS null_user_session_count,
        ROUND((SUM(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 4) AS null_user_session_pct,

        -- Event Time Nulls
        SUM(CASE WHEN event_time IS NULL THEN 1 ELSE 0 END) AS null_event_time_count,
        ROUND((SUM(CASE WHEN event_time IS NULL THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 4) AS null_event_time_pct,

        -- Price Nulls
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price_count,
        ROUND((SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 4) AS null_price_pct

    FROM arrow_table
"""

print("Scanning core columns for missing data...")
display(con.execute(null_check_query).df())

Scanning core columns for missing data...


,total_events,null_event_type_count,null_event_type_pct,null_user_session_count,null_user_session_pct,null_event_time_count,null_event_time_pct,null_price_count,null_price_pct
0,8829315,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
checkout_speed_query = """
    WITH cart_times AS (
        -- Find the exact moment they first added something to the cart
        SELECT
            user_session,
            MIN(event_time) AS first_cart_time
        FROM arrow_table
        WHERE event_type = 'cart'
        GROUP BY user_session
    ),
    purchase_times AS (
        -- Find the exact moment they completed the purchase
        SELECT
            user_session,
            MIN(event_time) AS first_purchase_time
        FROM arrow_table
        WHERE event_type = 'purchase'
        GROUP BY user_session
    ),
    checkout_deltas AS (
        -- Calculate the difference in seconds and minutes
        SELECT
            c.user_session,
            date_diff('second', c.first_cart_time, p.first_purchase_time) AS seconds_to_checkout,
            ROUND(date_diff('second', c.first_cart_time, p.first_purchase_time) / 60.0, 2) AS minutes_to_checkout
        FROM cart_times c
        JOIN purchase_times p ON c.user_session = p.user_session
        -- Ensure the purchase actually happened AFTER the cart add
        WHERE p.first_purchase_time >= c.first_cart_time
    )

    -- Aggregate to find the behavioral truth
    SELECT
        COUNT(*) AS total_converting_sessions,
        ROUND(AVG(minutes_to_checkout), 2) AS avg_minutes_to_checkout,
        MIN(seconds_to_checkout) AS fastest_checkout_seconds,
        MAX(minutes_to_checkout) AS slowest_checkout_minutes,

        -- How many are checking out suspiciously fast? (Under 10 seconds)
        SUM(CASE WHEN seconds_to_checkout < 10 THEN 1 ELSE 0 END) AS suspicious_fast_checkouts
    FROM checkout_deltas
"""

print("Executing Time-to-Checkout Analysis...")
display(con.execute(checkout_speed_query).df())

Executing Time-to-Checkout Analysis...


,total_converting_sessions,avg_minutes_to_checkout,fastest_checkout_seconds,slowest_checkout_minutes,suspicious_fast_checkouts
0,54499,3.09,6,2655.07,217.0


In [ ]:
null_dimension_query = """
    SELECT
        COUNT(*) AS total_rows,

        -- User ID Nulls
        SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_user_id_count,
        ROUND((SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2) AS null_user_id_pct,

        -- Product ID Nulls
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id_count,
        ROUND((SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2) AS null_product_id_pct,

        -- Category Code Nulls
        SUM(CASE WHEN category_id IS NULL THEN 1 ELSE 0 END) AS null_category_id_count,
        ROUND((SUM(CASE WHEN category_id IS NULL THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 2) AS null_category_id_pct

    FROM arrow_table
"""

print("Scanning dimensions for missing data...")
display(con.execute(null_dimension_query).df())

Scanning dimensions for missing data...


,total_rows,null_user_id_count,null_user_id_pct,null_product_id_count,null_product_id_pct,null_category_id_count,null_category_id_pct
0,8829315,0.0,0.0,0.0,0.0,0.0,0.0


# understand aggration table

In [ ]:
query = """
 SELECT
        user_session,
        user_id,
        MIN(event_time) AS session_start_time,
        MAX(event_time) AS session_end_time,
        MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,

        COUNT(CASE WHEN event_type = 'view' THEN 1 END) AS total_views,
        SUM(CASE WHEN event_type = 'cart' THEN price ELSE 0 END) AS gross_cart_value,
        COUNT(DISTINCT category_code) AS unique_categories_viewed,

        -- Creating flags for the target variable
        MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS triggered_cart,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS triggered_purchase
    FROM arrow_table
    GROUP BY user_session, user_id
    limit 5

"""

print("Scanning ...")
display(con.execute(query).df())

Scanning ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,user_session,user_id,session_start_time,session_end_time,first_cart_time,total_views,gross_cart_value,unique_categories_viewed,triggered_cart,triggered_purchase
0,0435dba2-a5f3-4954-a9d7-ef7cd7153af9,557234522,2019-10-06 04:44:34+00:00,2019-10-06 04:44:34+00:00,NaT,1,0.00,1,0,0
1,02700186-5b7e-4de5-ab17-bddf18c78c98,513225144,2019-10-06 08:32:44+00:00,2019-10-06 08:42:23+00:00,NaT,13,0.00,1,0,0
2,032583bd-7bdb-4c99-b549-6edc3d0f8346,552635981,2019-10-06 16:26:55+00:00,2019-10-06 16:29:25+00:00,2019-10-06 16:28:38+00:00,7,218.54,0,1,0
3,01ad8d91-0b22-4379-860f-5e802b7c4a7c,516232503,2019-10-06 03:43:37+00:00,2019-10-06 03:56:09+00:00,NaT,25,0.00,1,0,0
4,0612aecd-e70c-487d-a414-a565af7d2704,513841997,2019-10-06 09:29:33+00:00,2019-10-06 09:34:46+00:00,2019-10-06 09:33:48+00:00,11,8.88,1,1,0


In [ ]:
query = """
 with  session_base AS (
    SELECT
        user_session,
        user_id,
        MIN(event_time) AS session_start_time,
        MAX(event_time) AS session_end_time,
        MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,

        COUNT(CASE WHEN event_type = 'view' THEN 1 END) AS total_views,
        SUM(CASE WHEN event_type = 'cart' THEN price ELSE 0 END) AS gross_cart_value,
        COUNT(DISTINCT category_code) AS unique_categories_viewed,

        -- Creating flags for the target variable
        MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS triggered_cart,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS triggered_purchase
    FROM arrow_table
    GROUP BY user_session, user_id
)


    SELECT
        user_session,
        -- Counts chronological sessions per user. 1 = First Visit. 5 = Returning loyalist.
        ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY session_start_time) AS user_visit_number
    FROM session_base
    limit 7


"""

print("Scanning ...")
display(con.execute(query).df())

Scanning ...


,user_session,user_visit_number
0,884233e8-8b9f-4970-808b-4e1c81f8a5fc,1
1,cc654a5b-e561-4e6b-83aa-9b450a4f633c,1
2,b74568ee-fd16-4538-943c-0f2718b87fba,2
3,91769fdf-461b-4e43-9c73-88a07481b75c,1
4,ac5e27ca-6408-47ab-9837-19893a0334c2,2
5,345fe77b-e2df-4ad5-82f7-d3ec5867bc45,1
6,bae24929-9c22-44dd-a73e-b74eee651718,1


feature enginering

#  "How much money is actually sitting in abandoned carts?"


In [ ]:
query = """
 select*
 from arrow_table
 limit 3


"""

print("Scanning ...")
display(con.execute(query).df())

Scanning ...


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-05 15:18:46+00:00,view,1003428,2053013555631882655,electronics.smartphone,meizu,409.11,535521402,00141f1a-c8c5-4c26-a199-726bb0c0e11f
1,2019-10-05 17:15:15+00:00,view,1003878,2053013555631882655,electronics.smartphone,prestigio,51.46,541889611,02a6251c-c182-4667-aa32-ac01504c9989
2,2019-10-05 17:52:33+00:00,view,1004194,2053013555631882655,electronics.smartphone,tp-link,141.32,512709506,02bab9fc-e5b9-405c-b017-df4bd8b31cee


In [ ]:
query = """
WITH session_agg AS (
 SELECT
    user_session,

    -- flags
    MAX(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS has_view,
    MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS has_cart,
    MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS has_purchase,

    -- price aggregations
    SUM(CASE WHEN event_type = 'view' THEN price ELSE 0 END) AS view_price,
    SUM(CASE WHEN event_type = 'cart' THEN price ELSE 0 END) AS cart_price,
    SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END) AS purchase_price

FROM arrow_table
GROUP BY user_session
)

SELECT *
FROM session_agg
WHERE has_view = 1 AND has_cart = 0 AND has_purchase = 0
limit 10

"""

print("Scanning ...")
display(con.execute(query).df())

Scanning ...


,user_session,has_view,has_cart,has_purchase,view_price,cart_price,purchase_price
0,f20e94b4-3a94-4819-b30d-d6ac85c528bb,1,0,0,2209.39,0.0,0.0
1,f0e52332-30a7-4764-b2a9-90bc6d5768a3,1,0,0,3263.74,0.0,0.0
2,f493ad6a-e56d-44dc-bca1-cd34f30dd2c7,1,0,0,230.45,0.0,0.0
3,f5abe46c-90d7-4e28-b0e7-044fe234e3e0,1,0,0,1037.49,0.0,0.0
4,f0f02952-79ea-40f2-950f-44533e2b687c,1,0,0,342.61,0.0,0.0
5,f41cf9ae-9b4e-4996-a995-d672c897e386,1,0,0,1536.97,0.0,0.0
6,f90bfc8d-e510-4327-a644-07bdcad8320e,1,0,0,1377.40,0.0,0.0
7,fd5134f8-9c5e-44a9-bcd4-895cf4587953,1,0,0,5939.62,0.0,0.0
8,f0f1d0a5-c86f-4293-a74d-0c48200af2d5,1,0,0,3042.38,0.0,0.0
9,f536c2fe-7f2b-4de0-a9f8-cfe369cdf9a7,1,0,0,6233.75,0.0,0.0


In [ ]:
query = """
WITH session_agg AS (
 SELECT
    user_session,

    -- flags
    MAX(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS has_view,
    MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS has_cart,
    MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS has_purchase,

    -- price aggregations
    SUM(CASE WHEN event_type = 'view' THEN price ELSE 0 END) AS view_price,
    SUM(CASE WHEN event_type = 'cart' THEN price ELSE 0 END) AS cart_price,
    SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END) AS purchase_price

FROM arrow_table
GROUP BY user_session
)

SELECT *
FROM session_agg
WHERE has_view = 1 AND has_cart = 1 AND has_purchase = 0
limit 10

"""

print("Scanning ...")
display(con.execute(query).df())

Scanning ...


,user_session,has_view,has_cart,has_purchase,view_price,cart_price,purchase_price
0,029450c8-284e-46a3-ac18-54693ecbf2d3,1,1,0,3471.85,150.95,0.0
1,f9368642-8c3f-40d3-a745-4298970133ad,1,1,0,10249.14,607.63,0.0
2,fceff5fe-d598-4a5b-b26e-d4b56e20aa17,1,1,0,4192.07,98.54,0.0
3,f997bca2-addc-4e94-8aa3-dc3fe79d454f,1,1,0,3657.80,83.97,0.0
4,f6e629d7-8bb5-47c3-aa37-1f99273101a2,1,1,0,1156.96,21.09,0.0
5,eff0dd51-8513-47b3-9d76-97dc7df1645d,1,1,0,173.87,97.76,0.0
6,f0d38c1d-1812-4ac1-9575-bc4403a04a7e,1,1,0,1156.90,388.13,0.0
7,f5d38b1b-ae51-4fb9-9609-49663980ceb9,1,1,0,7061.68,1250.97,0.0
8,e9a6a208-1963-4b4b-aa7a-bd4a1b4a4d40,1,1,0,720.68,360.34,0.0
9,eadc74ce-c5f0-4dba-ae24-bf1ddc0d754b,1,1,0,3749.81,28.55,0.0


In [ ]:
query = """
WITH session_agg AS (
 SELECT
    user_session,

    -- flags
    MAX(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS has_view,
    MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS has_cart,
    MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS has_purchase,

    -- price aggregations
    SUM(CASE WHEN event_type = 'view' THEN price ELSE 0 END) AS view_price,
    SUM(CASE WHEN event_type = 'cart' THEN price ELSE 0 END) AS cart_price,
    SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END) AS purchase_price

FROM arrow_table
GROUP BY user_session
)

SELECT *
FROM session_agg
WHERE has_view = 1 AND has_cart = 1 AND has_purchase = 1
limit 10

"""

print("Scanning ...")
display(con.execute(query).df())

Scanning ...


,user_session,has_view,has_cart,has_purchase,view_price,cart_price,purchase_price
0,02c74100-ab64-4240-957f-9c03d2bb7a5f,1,1,1,395.37,263.58,131.79
1,f3f3e739-7247-4b4e-bfda-bd86f683ef7f,1,1,1,1219.14,462.40,462.40
2,f3a706fd-0289-46c9-ba0b-d7454514c102,1,1,1,2684.96,1342.48,1342.48
3,f0f7cf67-a6fd-4214-9f42-13ecc76ff994,1,1,1,2749.91,1111.37,815.38
4,0c54a950-6860-42f4-977b-15689282ee2d,1,1,1,4403.63,591.26,591.26
5,0519a53e-6c97-4e2f-8b68-e91167e71e88,1,1,1,4206.27,462.38,462.38
6,1be29027-a1da-4591-85cb-e7752af77894,1,1,1,1887.40,370.61,370.61
7,f2c7c333-ee5b-4300-a841-6e32bbd1dd90,1,1,1,3880.12,174.35,348.70
8,fec9d835-0c3e-4519-b6ce-952ba3514806,1,1,1,5042.08,1476.78,738.39
9,f4288751-6c0b-4614-9fcf-ae5dbf5c65da,1,1,1,6806.87,873.91,290.93


#original  my question code

In [ ]:
query = """
WITH session_agg AS (
    SELECT
        user_session,

        MAX(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS has_view,
        MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS has_cart,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS has_purchase,

        SUM(CASE WHEN event_type = 'view' THEN price ELSE 0 END) AS view_price,
        SUM(CASE WHEN event_type = 'cart' THEN price ELSE 0 END) AS cart_price,
        SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END) AS purchase_price

    FROM arrow_table
    GROUP BY user_session
)

SELECT *,
    CASE
        WHEN has_cart = 1 AND has_purchase = 0 THEN 'abandoned'
        WHEN has_cart = 1 AND has_purchase = 1 THEN 'converted'
        ELSE 'view_only'
    END AS session_type
FROM session_agg


"""
df_final = con.execute(query).df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
query = """
 select *
 from df_final
 where has_view= 1 and has_cart= 1 and has_purchase =0
 limit 10


"""


r= display(con.execute(query).df())
r

,user_session,has_view,has_cart,has_purchase,view_price,cart_price,purchase_price,session_type
0,0325b6b1-d50f-4f6e-ad79-8a5f701273da,1,1,0,547.76,47.62,0.0,abandoned
1,f33d8a97-c7a0-4c2d-8a1a-61b383212365,1,1,0,975.56,975.56,0.0,abandoned
2,fc43f87b-d74a-4b14-84ed-a247e3a25d06,1,1,0,2829.76,295.77,0.0,abandoned
3,019e9da4-639f-45d1-9b2e-fef79bc74982,1,1,0,197.82,989.10,0.0,abandoned
4,f0053f50-f66a-48b0-8250-eadc1a0e1f33,1,1,0,993.44,136.40,0.0,abandoned
5,fd5b7c9a-4fb0-4863-9dda-f99d35466761,1,1,0,273.70,506.26,0.0,abandoned
6,1c7f571f-1e74-46bf-a94e-66ca4dfa419a,1,1,0,1093.57,45.41,0.0,abandoned
7,1d04dcad-6c0a-497a-a324-4c5d59c7ae99,1,1,0,1759.09,111.70,0.0,abandoned
8,05413f80-4a77-496a-98a9-50f423cc2960,1,1,0,1595.80,398.95,0.0,abandoned
9,04e646cb-81fb-40cb-a93f-104941de596d,1,1,0,67.02,22.34,0.0,abandoned


# How much money is actually sitting in abandoned carts?"


In [ ]:
df_final[df_final['session_type'] == 'abandoned']['cart_price'].sum()

np.float64(28530836.32)

# how many user

In [ ]:
query = """
 select count(*)
 from df_final
 where has_view= 1 and has_cart= 1 and has_purchase =0

"""
r= display(con.execute(query).df())
r

,count_star()
0,53331


average  value

# The Margin Bleed: "Who are the 'Safe Buyers' we are accidentally giving discounts to?"


In [ ]:
query = """
 select*
 from arrow_table
limit 2



"""

print("Scanning ...")
result= display(con.execute(query).df())
result

Scanning ...


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-05 17:15:34+00:00,view,1003877,2053013555631882655,electronics.smartphone,prestigio,51.20,541889611,02a6251c-c182-4667-aa32-ac01504c9989
1,2019-10-05 22:38:06+00:00,view,1004014,2053013555631882655,electronics.smartphone,nokia,138.97,549945372,02ff4921-06d1-497b-af8e-17a45b3731d5


In [ ]:
query = """
SELECT *
FROM arrow_table
WHERE user_session IN (
    SELECT user_session
    FROM arrow_table
    WHERE event_type = 'purchase'
    LIMIT 20
)
ORDER BY user_session, event_time
LIMIT 50
"""


r= display(con.execute(query).df())
r

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-05 03:52:46+00:00,view,12300068,2053013556311359947,construction.tools.drill,None,59.90,547326487,00440449-8ae5-42a3-ae75-72b84e9af0fc
1,2019-10-05 03:53:11+00:00,view,3601421,2053013563810775923,appliances.kitchen.washer,lg,406.68,547326487,00440449-8ae5-42a3-ae75-72b84e9af0fc
2,2019-10-05 03:54:06+00:00,view,5100564,2053013553341792533,electronics.clocks,samsung,259.98,547326487,00440449-8ae5-42a3-ae75-72b84e9af0fc
3,2019-10-05 03:56:20+00:00,purchase,5100564,2053013553341792533,electronics.clocks,samsung,259.98,547326487,00440449-8ae5-42a3-ae75-72b84e9af0fc
4,2019-10-05 03:56:54+00:00,view,5100564,2053013553341792533,electronics.clocks,samsung,259.98,547326487,00440449-8ae5-42a3-ae75-72b84e9af0fc
5,2019-10-05 03:57:45+00:00,view,5000691,2053013566100866035,appliances.sewing_machine,chayka,92.64,547326487,00440449-8ae5-42a3-ae75-72b84e9af0fc
6,2019-10-05 03:58:43+00:00,purchase,5000691,2053013566100866035,appliances.sewing_machine,chayka,92.64,547326487,00440449-8ae5-42a3-ae75-72b84e9af0fc
7,2019-10-05 03:59:09+00:00,view,5000691,2053013566100866035,appliances.sewing_machine,chayka,92.64,547326487,00440449-8ae5-42a3-ae75-72b84e9af0fc
8,2019-10-05 17:20:42+00:00,view,1004873,2053013555631882655,electronics.smartphone,samsung,380.31,512434820,004cf503-d69a-421d-be7a-7af9eee94904
9,2019-10-05 17:22:53+00:00,view,12719632,2053013553559896355,None,kapsen,56.37,512434820,004cf503-d69a-421d-be7a-7af9eee94904


In [ ]:
query = """
SELECT
        user_session,
        MIN(event_time) AS session_start,
        MAX(event_time) AS session_end,
        COUNT(*) AS total_events,
        COUNT(DISTINCT category_code) AS unique_categories,
        COUNT(DISTINCT product_id) AS unique_products
    FROM arrow_table
    GROUP BY user_session

    limit 10
"""


r= display(con.execute(query).df())
r

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,user_session,session_start,session_end,total_events,unique_categories,unique_products
0,01c1b23a-4fc2-41dd-8b62-a379e00a5158,2019-10-05 09:33:03+00:00,2019-10-05 09:35:22+00:00,3,1,1
1,0188f6da-a2af-4897-8502-53cfb6f82bb7,2019-10-05 06:24:53+00:00,2019-10-05 06:28:09+00:00,5,1,3
2,0274c9ef-2fb8-4999-8dbd-5ef029a7ab92,2019-10-05 02:49:21+00:00,2019-10-05 02:54:37+00:00,7,5,6
3,01de467b-b851-4a93-87f5-a1de0a491e7b,2019-10-05 03:00:05+00:00,2019-10-05 03:08:39+00:00,5,1,1
4,02a61b36-9425-442a-99f9-68364c5737fe,2019-10-05 11:31:17+00:00,2019-10-05 11:35:54+00:00,8,0,3
5,0129738b-dc04-4431-ad33-ed133f1cfa29,2019-10-05 15:30:44+00:00,2019-10-05 15:32:25+00:00,3,1,2
6,02f242f6-cf59-4920-9c09-3f105417762d,2019-10-05 17:28:36+00:00,2019-10-05 17:31:37+00:00,4,0,4
7,018f69c0-2202-47b2-81bc-edbce625b421,2019-10-05 22:14:42+00:00,2019-10-05 22:17:34+00:00,3,1,3
8,022db2ff-39b9-4e5a-8c6f-5d78ad853d06,2019-10-05 18:23:42+00:00,2019-10-05 18:37:55+00:00,18,4,17
9,0224c54d-2c82-4435-bd35-44a944205096,2019-10-05 13:36:46+00:00,2019-10-05 13:36:46+00:00,1,1,1


In [ ]:
# Your query
query = """
SELECT *
FROM arrow_table
WHERE user_session = 'f4793f55-de4e-45aa-8570-62c21f9b5db8'
ORDER BY user_session, event_time

"""

# Run query
df = con.execute(query).df()
df

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 08:46:08+00:00,view,10900328,2053013555069845885,appliances.kitchen.mixer,dauscher,12.84,515445705,f4793f55-de4e-45aa-8570-62c21f9b5db8
1,2019-10-01 08:48:57+00:00,view,3100105,2053013555262783879,appliances.kitchen.blender,braun,100.36,515445705,f4793f55-de4e-45aa-8570-62c21f9b5db8
2,2019-10-01 08:50:30+00:00,view,3100105,2053013555262783879,appliances.kitchen.blender,braun,100.36,515445705,f4793f55-de4e-45aa-8570-62c21f9b5db8
3,2019-10-01 08:51:22+00:00,view,3100105,2053013555262783879,appliances.kitchen.blender,braun,100.36,515445705,f4793f55-de4e-45aa-8570-62c21f9b5db8
4,2019-10-01 08:52:47+00:00,view,3100570,2053013555262783879,appliances.kitchen.blender,polaris,79.54,515445705,f4793f55-de4e-45aa-8570-62c21f9b5db8
5,2019-10-01 08:53:01+00:00,view,3100570,2053013555262783879,appliances.kitchen.blender,polaris,79.54,515445705,f4793f55-de4e-45aa-8570-62c21f9b5db8
6,2019-10-01 08:53:28+00:00,view,3100570,2053013555262783879,appliances.kitchen.blender,polaris,79.54,515445705,f4793f55-de4e-45aa-8570-62c21f9b5db8
7,2019-10-01 08:53:39+00:00,view,3100251,2053013555262783879,appliances.kitchen.blender,braun,82.34,515445705,f4793f55-de4e-45aa-8570-62c21f9b5db8
8,2019-10-01 08:56:03+00:00,view,3100251,2053013555262783879,appliances.kitchen.blender,braun,82.34,515445705,f4793f55-de4e-45aa-8570-62c21f9b5db8
9,2019-10-01 08:56:06+00:00,view,3100570,2053013555262783879,appliances.kitchen.blender,polaris,79.54,515445705,f4793f55-de4e-45aa-8570-62c21f9b5db8


In [ ]:
import duckdb
import pandas as pd
from IPython.display import FileLink

# Connect (skip if already defined)
con = duckdb.connect()

# Your query
query = """
SELECT *
FROM arrow_table
WHERE user_session = 'f4793f55-de4e-45aa-8570-62c21f9b5db8'
ORDER BY user_session, event_time
"""

# Run query
df = con.execute(query).df()

# Save locally
file_name = "output.csv"
df.to_csv(file_name, index=False)

# Generate download link
FileLink(file_name)

/content/output.csv

In [ ]:
import os
print(os.getcwd())

/content


In [ ]:
df.to_csv("/content/output.csv", index=False)

In [ ]:
import os
print(os.listdir("/content"))

['output.csv']


In [ ]:
from google.colab import files
files.download("/content/output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
query = """
WITH session_events AS (
    SELECT
        user_session,
        MIN(event_time) AS session_start,
        MAX(event_time) AS session_end,
        COUNT(*) AS total_events,
        COUNT(DISTINCT category_code) AS unique_categories,
        COUNT(DISTINCT product_id) AS unique_products
    FROM arrow_table
    GROUP BY user_session
),
 product_views AS (
    SELECT
        user_session,
        product_id,
        COUNT(*) AS view_count
    FROM arrow_table
    WHERE event_type = 'view'
    GROUP BY user_session, product_id
),

repeat_views AS (
    SELECT
        user_session,
        COUNT(*) AS repeated_products
    FROM product_views
    WHERE view_count > 1
    GROUP BY user_session
)
, cart_purchase_time AS (
    SELECT
        user_session,
        MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS cart_time,
        MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS purchase_time
    FROM arrow_table
    GROUP BY user_session
)

SELECT
    s.user_session,
    (s.session_end - s.session_start) AS session_duration,
    s.unique_categories,
    r.repeated_products,
    (c.purchase_time - c.cart_time) AS cart_to_purchase_delay
FROM session_events s
LEFT JOIN repeat_views r ON s.user_session = r.user_session
LEFT JOIN cart_purchase_time c ON s.user_session = c.user_session
WHERE c.purchase_time IS NOT NULL
LIMIT 50

"""


r= display(con.execute(query).df())
r

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,user_session,session_duration,unique_categories,repeated_products,cart_to_purchase_delay
0,f2c7c333-ee5b-4300-a841-6e32bbd1dd90,0 days 00:19:33,1,2,0 days 00:02:13
1,fd7153b3-b54a-4056-b0cd-28f30e8a6cb6,0 days 00:18:18,3,6,-1 days +23:51:52
2,f7f642d4-884d-4c2a-9fb2-cc1b21779c00,0 days 00:05:01,0,1,NaT
3,f4288751-6c0b-4614-9fcf-ae5dbf5c65da,0 days 02:23:36,2,5,0 days 00:08:53
4,fd5b8cf3-ae80-43ed-abd2-5f06f748812a,0 days 00:04:35,1,1,NaT
5,f44a08ee-2d59-4adf-afa6-1c79d4eb612d,0 days 00:02:14,1,1,NaT
6,f86546ae-1872-4227-a5f0-bf98d6aa4f79,0 days 00:04:44,1,1,0 days 00:01:41
7,05b87b98-e749-468e-8869-befd5c08b46a,0 days 00:01:07,1,1,NaT
8,1237819e-61a0-4502-8228-d414f91da96c,0 days 00:16:20,0,3,NaT
9,1d8eabca-f346-4d41-89b5-e4a7513d60bd,0 days 00:14:10,0,4,NaT
